In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html#sklearn.ensemble.HistGradientBoostingRegressor

# df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df = pd.read_csv('../../datasets/MX_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,5Z75FvRFiultmPFWHx5jQ7,7 Dias,"Gabito Ballesteros, Tito Double P",1,0,1,MX,2025-02-17,85,True,...,-3.749,1,0.0614,0.3860,0.000000,0.0839,0.577,111.913,3,Higher
1,78HEzDEs1QUnHB2DbxgC1s,Te Quería Ver,"Alemán, Neton Vega",2,1,1,MX,2025-02-17,82,False,...,-5.182,0,0.0681,0.1880,0.000017,0.0922,0.448,100.019,4,About_Average
2,0LTwdL5yZ6YOTEGUQPFuSN,ROSONES,Tito Double P,3,1,1,MX,2025-02-17,88,True,...,-5.939,1,0.0318,0.7040,0.000010,0.1170,0.604,120.129,3,Lower
3,7sd6zMrgGpEa7NkQm9TRrg,NADIE,Tito Double P,4,1,2,MX,2025-02-17,87,True,...,-4.710,1,0.1140,0.4650,0.000000,0.1200,0.526,92.604,4,About_Average
4,4eLDmhsJW3JoZTXCAozHor,Loco,Neton Vega,5,-3,45,MX,2025-02-17,61,False,...,-5.502,1,0.0686,0.0741,0.007680,0.1390,0.636,91.981,4,Lower


In [2]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [3]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor()
)

# Tuning Hyperparameters

In [4]:
def explore_single_hp_values(pipeline, param_name, param_values, X_train, y_train, k_fold, scoring):
    param_grid = {param_name: param_values}
    grid_search = GridSearchCV(pipeline, param_grid, cv=k_fold, scoring=scoring)
    grid_search.fit(X_train, y_train)
    result_columns = [f"param_{param_name}", "mean_test_score", "std_test_score", "rank_test_score"]
    return pd.DataFrame(grid_search.cv_results_)[result_columns]
  
import seaborn as sns
import matplotlib.pyplot as plt

def plot_gridsearch_heatmap(results_df, x_param, y_param, score='neg_mean_squared_error'):    
    # Pivot the table to format it for a heatmap
    heatmap_data = results_df.pivot(index=f'param_{y_param}', columns=f'param_{x_param}', values=score)
    
    # Plot heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_data, annot=True, cmap='viridis', fmt='.3f', linewidths=0.5)
    plt.title(f'Grid Search Results: {score}')
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.show()

In [5]:
# Hyperparameter Tuning
hgbr_step_name = pipeline.steps[1][0]
step_prefix = '__'
hgbr_step_prefix = f"{hgbr_step_name}{step_prefix}"

# Parameter Keys
loss_key = hgbr_step_prefix + "loss"
max_leaf_key = hgbr_step_prefix + "max_leaf_nodes"
max_depth_key = hgbr_step_prefix + "max_depth"

# hgbr_param_keys = HistGradientBoostingRegressor().get_params().keys()
# for key in [loss_key, learning_rate_key, max_leaf_key, max_depth_key]:
#     assert key.startswith(hgbr_step_name), f"Key {key} should start with {hgbr_step_name}"
#     assert "__" in key, f"Key {key} should be connected by a double underscope __"
#     assert key[key.index("__") + 2:] in hgbr_param_keys, f"Key {key} should end with a DT parameter name"

from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

k_fold = KFold(n_splits=5, shuffle=True, random_state=42)

# Define the param_grid and grid_search
param_grid = {
  loss_key: ['squared_error', 'absolute_error'],
  max_leaf_key: [ 40, 45, 50, 55],
  max_depth_key:[55, 65, 75, 85, 95],

}
grid_search = GridSearchCV(pipeline, param_grid, scoring="neg_mean_squared_error", cv=k_fold,   verbose= 3,)
grid_search.fit(X_train, y_train)

# Print the best parameters
grid_search.best_params_
grid_search.best_score_

Fitting 5 folds for each of 40 candidates, totalling 200 fits
[CV 1/5] END histgradientboostingregressor__loss=squared_error, histgradientboostingregressor__max_depth=55, histgradientboostingregressor__max_leaf_nodes=40;, score=-43.186 total time=   0.5s
[CV 2/5] END histgradientboostingregressor__loss=squared_error, histgradientboostingregressor__max_depth=55, histgradientboostingregressor__max_leaf_nodes=40;, score=-32.939 total time=   0.5s
[CV 3/5] END histgradientboostingregressor__loss=squared_error, histgradientboostingregressor__max_depth=55, histgradientboostingregressor__max_leaf_nodes=40;, score=-32.737 total time=   0.6s
[CV 4/5] END histgradientboostingregressor__loss=squared_error, histgradientboostingregressor__max_depth=55, histgradientboostingregressor__max_leaf_nodes=40;, score=-27.434 total time=   0.4s
[CV 5/5] END histgradientboostingregressor__loss=squared_error, histgradientboostingregressor__max_depth=55, histgradientboostingregressor__max_leaf_nodes=40;, score=

np.float64(-33.902021485717775)

In [7]:
# results = pd.DataFrame(grid_search.cv_results_)
# results = results[results['param_' + loss_key] == 'squared_error']
# plot_gridsearch_heatmap(results, max_leaf_key, max_depth_key)
grid_search.best_params_

{'histgradientboostingregressor__loss': 'squared_error',
 'histgradientboostingregressor__max_depth': 65,
 'histgradientboostingregressor__max_leaf_nodes': 45}

In [8]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Pre-tuned")
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

Pre-tuned
MSE:  31.938524750793977
RMSE:  5.651417941613766


# Fitting Pipeline w/ Tuned HPs

In [9]:
pipeline_tuned = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor(loss='squared_error', max_depth=65, max_leaf_nodes=45)
)

pipeline_tuned.fit(X_train, y_train)
y_pred = pipeline_tuned.predict(X_test)

print("Tuned!")
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

Tuned!
MSE:  32.956401243120915
RMSE:  5.740766607616174


In [10]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())

count    4063.000000
mean        1.172040
std         0.791813
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        20.000000
Name: count, dtype: float64
count     56.000000
mean      85.035714
std      118.018175
min        1.000000
25%        3.500000
50%       27.500000
75%      104.750000
max      391.000000
Name: count, dtype: float64
